In [1]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr
!apt-get install -y -qq tesseract-ocr-hin
!apt-get install -y -qq tesseract-ocr-asm

!pip install -q opencv-python-headless pytesseract pandas openpyxl pillow matplotlib

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-hin.
(Reading database ... 122806 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-hin_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-hin (1:4.1.0-2) ...
Setting up tesseract-ocr-hin (1:4.1.0-2) ...
Selecting previously unselected package tesseract-ocr-asm.
(Reading database ... 122810 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-asm_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-asm (1:4.1.0-2) ...
Setting up tesseract-ocr-asm (1:4.1.0-2) ...


In [20]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATASET_DIR = "/content/drive/MyDrive/FloodSense_OCR_NLP_Dataset/images"

if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_DIR}")

print("Dataset folder found!")
print("Files:", len(os.listdir(DATASET_DIR)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset folder found!
Files: 300


In [21]:
import os
import cv2
import pandas as pd
import pytesseract
from langdetect import detect,DetectorFactory,LangDetectException
from difflib import SequenceMatcher
DetectorFactory=0

# Look for the ground truth file in the parent directory of DATASET_DIR
GROUND_TRUTH_PATH = os.path.join(
    os.path.dirname(DATASET_DIR),
    "FloodSense_Dataset_Ground_Truth_300.csv"
)

if not os.path.exists(GROUND_TRUTH_PATH):
    # Fallback to current DATASET_DIR if not found in parent directory
    GROUND_TRUTH_PATH = os.path.join(
        DATASET_DIR,
        "FloodSense_Dataset_Ground_Truth_300.csv"
    )

if not os.path.exists(GROUND_TRUTH_PATH):
    raise FileNotFoundError(f"Could not find ground truth file at {GROUND_TRUTH_PATH} or in its parent directory.")

ground_truth = pd.read_csv(GROUND_TRUTH_PATH)

required_columns = ["Image_Name", "Original_Text"]

for column in required_columns:
    if column not in ground_truth.columns:
        raise ValueError(f"Missing required column: {column}")

print("Ground-truth records:", len(ground_truth))
print("Installed Tesseract languages:", pytesseract.get_languages(config=""))
print("Image files:", sum(
    f.lower().endswith((".png", ".jpg", ".jpeg"))
    for f in os.listdir(DATASET_DIR)
))

display(ground_truth.head())

Ground-truth records: 300
Installed Tesseract languages: ['asm', 'eng', 'hin', 'osd']
Image files: 300


,Image_Name,Language,Original_Text,Expected_Needs,Expected_Keywords,Expected_Location,Expected_People_Count,Expected_Urgency_Level,Image_Type
0,FS_EN_001.png,EN,Water has reduced in Dhemaji since morning. No...,information,rescue;evacuation,Dhemaji,0,Low,Printed_Report
1,FS_EN_002.png,EN,DAMAGE REPORT\nMud and flood water have damage...,information,flood water;bridge,Ballia,0,Medium,Printed_Report
2,FS_EN_003.png,EN,Embankment breached near Barpeta Road at 8 PM....,evacuation,embankment;breached;evacuate,Barpeta Road,60,Critical,Emergency_Message
3,FS_EN_004.png,EN,Request: 40 flood victims in Madhubani need me...,medical,flood;medicines;fever;diarrhoea;doctor,Madhubani,40,High,Community_Request
4,FS_EN_005.png,EN,Our village in Darbhanga has no electricity an...,blankets_clothing;shelter,blankets;tarpaulin;urgently,Darbhanga,60,High,Community_Request


In [22]:
import re

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def similarity_score(expected, extracted):
    expected = normalize_text(expected)
    extracted = normalize_text(extracted)

    if not expected:
        return 100.0 if not extracted else 0.0

    return round(
        SequenceMatcher(
            None,
            expected,
            extracted
        ).ratio() * 100,
        2
    )

In [23]:
%pip install langdetect

In [24]:
LANGUAGE_MAP = {
    "EN": ["eng"],
    "HI": ["hin"],
    "AS": ["asm"]
}


def detect_language(text):
  text=str(text).strip()
  if len(text) <10:
    return "unknown"
  try:
    detected_code=detect(text)
    return detected_code
  except LangDetectException:
    return "unknown"


def preprocess_image(gray, method):
    if method == "original":
        return gray

    if method == "otsu":
        _, result = cv2.threshold(
            gray,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        return result

    if method == "adaptive":
        block_size = 31

        if min(gray.shape) < block_size:
            block_size = max(
                3,
                min(gray.shape) // 2 * 2 - 1
            )

        return cv2.adaptiveThreshold(
            gray,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            block_size,
            11
        )

    raise ValueError(f"Unknown preprocessing method: {method}")


def run_tesseract(image, language, psm):
    config = f"--oem 3 --psm {psm}"

    return pytesseract.image_to_string(
        image,
        lang=language,
        config=config
    ).strip()

In [25]:
LANGUAGE_CODE_MAP = {
    "en": "eng",
    "hi": "hin",
    "as": "asm",
    "unknown": "unknown"
}

def adaptive_ocr(image_path, language_options):
    image = cv2.imread(image_path)

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    image = cv2.resize(
        image,
        None,
        fx=2,
        fy=2,
        interpolation=cv2.INTER_CUBIC
    )

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    candidates = []

    # Full-image OCR
    for method in ["original", "otsu", "adaptive"]:
        processed = preprocess_image(gray, method)

        for language in language_options:
            for psm in [3, 6, 11]:
                try:
                    text = run_tesseract(
                        processed,
                        language,
                        psm
                    )

                    detected_language = detect_language(text)

                    candidates.append({
                        "Layout": "full_image",
                        "Preprocessing": method,
                        "Language": language,
                        "Detected_Language": detected_language,
                        "PSM": psm,
                        "Extracted_Text": text
                    })

                except pytesseract.TesseractError as error:
                    print(
                        f"Skipped {method}/{language}/PSM{psm}: {error}"
                    )

    # Two-column OCR
    height, width = gray.shape
    midpoint = width // 2
    gap = max(1, int(width * 0.015))

    left = gray[:, :max(1, midpoint - gap)]
    right = gray[:, min(width, midpoint + gap):]

    if right.size > 0:
        for method in ["original", "otsu"]:
            left_processed = preprocess_image(left, method)
            right_processed = preprocess_image(right, method)

            for language in language_options:
                try:
                    left_text = run_tesseract(
                        left_processed,
                        language,
                        6
                    )

                    right_text = run_tesseract(
                        right_processed,
                        language,
                        6
                    )

                    text = (
                        left_text + "\n" + right_text
                    ).strip()

                    detected_language = detect_language(text)

                    candidates.append({
                        "Layout": "two_columns",
                        "Preprocessing": method,
                        "Language": language,
                        "Detected_Language": detected_language,
                        "PSM": 6,
                        "Extracted_Text": text
                    })

                except pytesseract.TesseractError as error:
                    print(
                        f"Skipped column OCR {method}/{language}: {error}"
                    )

    return candidates

results = []
LANGUAGE_OPTIONS = ["eng", "hin", "asm"]

for _, row in ground_truth.iterrows():
    image_name = str(row["Image_Name"])
    expected_text = str(row["Original_Text"])

    image_path = os.path.join(DATASET_DIR, image_name)

    if not os.path.exists(image_path):
        print("Missing image:", image_name)
        continue

    try:
        candidates = adaptive_ocr(
            image_path,
            LANGUAGE_OPTIONS
        )

        if not candidates:
            print("No OCR candidates generated:", image_name)
            continue

        # Evaluate every OCR candidate against ground truth
        for candidate in candidates:
            candidate["Image_Name"] = image_name

            candidate["Similarity_Percent"] = similarity_score(
                expected_text,
                candidate["Extracted_Text"]
            )

            # Determine whether the language detector agrees with Tesseract
            expected_language = candidate["Language"]
            detected_tesseract_language = LANGUAGE_CODE_MAP.get(
                candidate["Detected_Language"],
                "unknown"
            )

            candidate["Language_Match"] = (
                detected_tesseract_language == expected_language
            )

        # Select the OCR candidate with the highest text similarity
        best = max(
            candidates,
            key=lambda item: item["Similarity_Percent"]
        )

        results.append(best)

        print(
            f"{image_name}: "
            f"{best['Similarity_Percent']:.2f}% | "
            f"{best['Layout']} | "
            f"{best['Preprocessing']} | "
            f"OCR={best['Language']} | "
            f"Detected={best['Detected_Language']} | "
            f"PSM {best['PSM']}"
        )

    except Exception as error:
        print(f"Error processing {image_name}: {error}")

results_df = pd.DataFrame(results)
print("\nImages evaluated:", len(results_df))

if not results_df.empty:
    print(
        "Mean text similarity:",
        round(results_df["Similarity_Percent"].mean(), 2),
        "%"
    )

    print(
        "Language agreement:",
        round(results_df["Language_Match"].mean() * 100, 2),
        "%"
    )

    display(
        results_df[
            [
                "Image_Name",
                "Similarity_Percent",
                "Layout",
                "Preprocessing",
                "Language",
                "Detected_Language",
                "Language_Match",
                "PSM"
            ]
        ].sort_values("Image_Name")
    )

FS_EN_001.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 3
FS_EN_002.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 3
FS_EN_003.png: 99.32% | full_image | original | OCR=eng | Detected=en | PSM 3
FS_EN_004.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 6
FS_EN_005.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 3
FS_EN_006.png: 99.29% | full_image | adaptive | OCR=eng | Detected=en | PSM 6
FS_EN_007.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 3
FS_EN_008.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 11
FS_EN_009.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 3
FS_EN_010.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 3
FS_EN_011.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 3
FS_EN_012.png: 100.00% | full_image | original | OCR=eng | Detected=en | PSM 11
FS_EN_013.png: 100.00% | full_image | original | OCR

,Image_Name,Similarity_Percent,Layout,Preprocessing,Language,Detected_Language,Language_Match,PSM
200,FS_AS_001.png,99.26,full_image,otsu,asm,bn,False,11
201,FS_AS_002.png,99.28,full_image,adaptive,asm,bn,False,3
202,FS_AS_003.png,96.77,full_image,otsu,asm,bn,False,3
203,FS_AS_004.png,96.27,full_image,original,asm,bn,False,6
204,FS_AS_005.png,99.06,full_image,adaptive,asm,bn,False,3
...,...,...,...,...,...,...,...,...
195,FS_HI_096.png,100.00,full_image,original,hin,hi,True,3
196,FS_HI_097.png,100.00,full_image,adaptive,hin,hi,True,3
197,FS_HI_098.png,99.07,full_image,original,hin,hi,True,3
198,FS_HI_099.png,97.62,full_image,original,hin,hi,True,3
